# Inspect Political-Corruption Classifier Comparison

Use this notebook after running `political_classifier/scripts/compare_models.py`. The script does the heavy/reproducible model comparison; this notebook reads the saved CSV files and makes the results easier to inspect.

## 1. Load Comparison Outputs

In [ ]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "config.py").exists() and (path / "dataloader.py").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing config.py and dataloader.py")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

COMPARISON_DIR = Path(
    "/home/akroon/data/1t_storage/RESPOND-victims-of-corruption/"
    "political_corruption_pipeline/classifier_comparison"
)

best_results_path = COMPARISON_DIR / "best_model_results.csv"
threshold_results_path = COMPARISON_DIR / "all_threshold_results.csv"
country_results_path = COMPARISON_DIR / "country_results_for_best_silver_thresholds.csv"
predictions_path = COMPARISON_DIR / "validation_prediction_comparison.csv"

for path in [best_results_path, threshold_results_path, country_results_path, predictions_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. First run: python3 political_classifier/scripts/compare_models.py")

best_results = pd.read_csv(best_results_path)
threshold_results = pd.read_csv(threshold_results_path)
country_results = pd.read_csv(country_results_path)
predictions = pd.read_csv(predictions_path)

print(f"Loaded comparison outputs from: {COMPARISON_DIR}")
print(f"Best-result rows: {len(best_results):,}")
print(f"Threshold rows:    {len(threshold_results):,}")
print(f"Country rows:      {len(country_results):,}")
print(f"Prediction rows:   {len(predictions):,}")

## 2. Select Comparison Run

By default, this notebook uses only the current final comparison folder: `classifier_comparison`. Older folders such as `classifier_comparison_uk_calibration` may still exist on disk, but they are intentionally ignored so archived experiments do not leak back into the final tables.


In [ ]:
BASE_DIR = Path(
    "/home/akroon/data/1t_storage/RESPOND-victims-of-corruption/"
    "political_corruption_pipeline"
)

# Keep this explicit. Do not glob classifier_comparison*, because old archived
# output folders can still exist on disk and should not enter final tables.
COMPARISON_RUNS = {
    "comparison": BASE_DIR / "classifier_comparison",
}

comparison_dirs = {
    run_name: folder
    for run_name, folder in COMPARISON_RUNS.items()
    if folder.is_dir()
}

if not comparison_dirs:
    raise FileNotFoundError(
        f"No selected classifier comparison folders found. Expected: {COMPARISON_RUNS}"
    )

print("Comparison folders selected:")
for run_name, folder in comparison_dirs.items():
    print(f"- {run_name}: {folder}")

best_frames = []
threshold_frames = []
country_frames = []
prediction_frames = []

for run_name, folder in comparison_dirs.items():
    best_path = folder / "best_model_results.csv"
    threshold_path = folder / "all_threshold_results.csv"
    country_path = folder / "country_results_for_best_silver_thresholds.csv"
    prediction_path = folder / "validation_prediction_comparison.csv"

    if not best_path.exists():
        print(f"Skipping {run_name}; missing {best_path.name}")
        continue

    data = pd.read_csv(best_path)
    data["comparison_run"] = run_name
    best_frames.append(data)

    if threshold_path.exists():
        data = pd.read_csv(threshold_path)
        data["comparison_run"] = run_name
        threshold_frames.append(data)
    else:
        print(f"Missing threshold table for {run_name}: {threshold_path}")

    if country_path.exists():
        data = pd.read_csv(country_path)
        data["comparison_run"] = run_name
        country_frames.append(data)
    else:
        print(f"Missing country table for {run_name}: {country_path}")

    if prediction_path.exists():
        data = pd.read_csv(prediction_path)
        data["comparison_run"] = run_name
        prediction_frames.append(data)
    else:
        print(f"Missing prediction table for {run_name}: {prediction_path}")

inspection_best_results = pd.concat(best_frames, ignore_index=True)
inspection_threshold_results = pd.concat(threshold_frames, ignore_index=True)
inspection_country_results = pd.concat(country_frames, ignore_index=True) if country_frames else pd.DataFrame()
inspection_prediction_results = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()

print(
    f"Loaded {len(inspection_best_results):,} best-result rows "
    f"across {len(comparison_dirs):,} selected comparison run(s)."
)


## 3. Best Model Table

Sort by political-corruption F1 first. Also check precision, recall, and predicted-positive rate before choosing a final classifier.

In [ ]:
display_columns = [
    "comparison_run",
    "model",
    "embedding_model",
    "label_source",
    "train_rows",
    "threshold",
    "accuracy",
    "political_precision",
    "political_recall",
    "political_f1",
    "macro_f1",
    "weighted_f1",
    "predicted_positive_rate",
]

best_display = (
    inspection_best_results[display_columns]
    .sort_values(["political_f1", "political_recall", "political_precision"], ascending=False)
    .reset_index(drop=True)
)

best_display


## 4. Recommended Final Candidate

This cell recommends the best `silver_combined` model across all discovered comparison runs. We keep this as the conservative final political-corruption classifier; the UK-calibrated variant is reported as a sensitivity check.

In [ ]:
silver_combined = inspection_best_results[
    inspection_best_results["label_source"].eq("silver_combined")
].copy()
silver_combined = silver_combined.sort_values(
    ["political_f1", "political_recall", "political_precision"],
    ascending=False,
)

recommended = silver_combined.iloc[0]

print("Recommended conservative final political-corruption classifier")
print(f"Comparison run: {recommended.get('comparison_run', '')}")
print(f"Embedding model: {recommended['embedding_model']}")
print(f"Threshold:       {recommended['threshold']}")
print(f"Political F1:    {recommended['political_f1']:.3f}")
print(f"Precision:       {recommended['political_precision']:.3f}")
print(f"Recall:          {recommended['political_recall']:.3f}")
print(f"Positive rate:   {recommended['predicted_positive_rate']:.2%}")

silver_combined[display_columns]


## 5. Threshold Sweep For A Candidate

In [ ]:
SELECTED_EMBEDDING_MODEL = recommended["embedding_model"]
SELECTED_LABEL_SOURCE = recommended["label_source"]
SELECTED_COMPARISON_RUN = recommended.get("comparison_run", None)

candidate_thresholds = inspection_threshold_results[
    inspection_threshold_results["embedding_model"].fillna("").eq(str(SELECTED_EMBEDDING_MODEL))
    & inspection_threshold_results["label_source"].eq(SELECTED_LABEL_SOURCE)
].copy()

if SELECTED_COMPARISON_RUN is not None and "comparison_run" in candidate_thresholds.columns:
    candidate_thresholds = candidate_thresholds[
        candidate_thresholds["comparison_run"].eq(SELECTED_COMPARISON_RUN)
    ].copy()

candidate_thresholds[
    [
        "threshold",
        "accuracy",
        "political_precision",
        "political_recall",
        "political_f1",
        "macro_f1",
        "weighted_f1",
        "predicted_positive_rate",
    ]
]


In [ ]:
ax = candidate_thresholds.plot(
    x="threshold",
    y=["political_precision", "political_recall", "political_f1"],
    marker="o",
    figsize=(8, 4),
)
ax.set_ylim(0, 1)
ax.set_title(f"Threshold sweep: {SELECTED_EMBEDDING_MODEL} / {SELECTED_LABEL_SOURCE}")
ax.set_ylabel("Score")
ax.grid(True, alpha=0.3)

## 6. Country-Level Validation

In [ ]:
candidate_country = inspection_country_results[
    inspection_country_results["embedding_model"].fillna("").eq(str(SELECTED_EMBEDDING_MODEL))
    & inspection_country_results["label_source"].eq(SELECTED_LABEL_SOURCE)
].copy()

if SELECTED_COMPARISON_RUN is not None and "comparison_run" in candidate_country.columns:
    candidate_country = candidate_country[
        candidate_country["comparison_run"].eq(SELECTED_COMPARISON_RUN)
    ].copy()

candidate_country = candidate_country.sort_values("political_f1", ascending=False)

candidate_country[
    [
        "country",
        "n",
        "political_support",
        "accuracy",
        "political_precision",
        "political_recall",
        "political_f1",
        "predicted_positive_rate",
    ]
]


In [ ]:
ax = candidate_country.sort_values("political_f1").plot.barh(
    x="country",
    y="political_f1",
    figsize=(8, 5),
    legend=False,
)
ax.set_xlim(0, 1)
ax.set_title("Political-corruption F1 by country")
ax.set_xlabel("F1")
ax.grid(True, axis="x", alpha=0.3)

## 7. LaTeX Tables For Political-Corruption Classification

This section writes compact LaTeX tables for the manuscript and more detailed tables for the appendix. All tables are explicitly labelled as political-corruption classification results, because later workflow steps may add other classification tasks. The generated tables require `\usepackage{booktabs}` and `\usepackage{graphicx}` in the manuscript preamble.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

TABLE_DIR = BASE_DIR / "manuscript_tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

LATEX_TABLE_NOTE = (
    "Note. PC = political corruption. Precision, recall, and PC F1 refer to the "
    "political-corruption class."
)

def model_label(row):
    if row["model"] == "tfidf_char_ngrams_logreg":
        return "TF-IDF char. n-grams"
    embedding = str(row.get("embedding_model", ""))
    if embedding == "intfloat/multilingual-e5-large":
        return "E5-large"
    if embedding == "intfloat/multilingual-e5-base":
        return "E5-base"
    if embedding == "BAAI/bge-m3":
        return "BGE-M3"
    if embedding == "sentence-transformers/LaBSE":
        return "LaBSE"
    if embedding == "sentence-transformers/paraphrase-multilingual-mpnet-base-v2":
        return "mMPNet"
    if embedding == "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2":
        return "mMiniLM"
    return embedding or row["model"]

def label_source_label(value):
    labels = {
        "silver_combined": "Silver 1+2",
        "silver_batch_1": "Silver 1",
        "silver_batch_2": "Silver 2",
        "human_5fold_cv": "Human CV",
    }
    return labels.get(value, value)

def resize_latex_tabular(latex):
    """Wrap the tabular environment so wide APA manuscript tables fit."""
    begin = "\\begin{tabular}"
    end = "\\end{tabular}"
    if begin not in latex or end not in latex:
        return latex
    latex = latex.replace(begin, "\\resizebox{\\textwidth}{!}{%\n" + begin, 1)
    latex = latex.replace(end, end + "\n}", 1)
    return latex

def add_table_note(latex, note=LATEX_TABLE_NOTE):
    return latex.replace("\\end{table}\n", f"\\par\\smallskip\\footnotesize{{{note}}}\n\\end{{table}}\n")

def save_latex_table(dataframe, filename, caption, label, float_format="%.3f", resize=True, note=True):
    path = TABLE_DIR / filename
    latex = dataframe.to_latex(
        index=False,
        escape=True,
        caption=caption,
        label=label,
        float_format=float_format,
        bold_rows=False,
    )
    if resize:
        latex = resize_latex_tabular(latex)
    if note:
        latex = add_table_note(latex)
    path.write_text(latex)
    print(f"Saved {path}")
    return path

# Main manuscript table: final model plus key baselines/alternatives.
main_rows = inspection_best_results[
    inspection_best_results["label_source"].isin(["silver_combined", "human_5fold_cv"])
].copy()
main_rows["model_family"] = main_rows.apply(model_label, axis=1)
main_rows["training_labels"] = main_rows["label_source"].map(label_source_label)

preferred_order = [
    "E5-large",
    "BGE-M3",
    "E5-base",
    "LaBSE",
    "mMPNet",
    "mMiniLM",
    "TF-IDF char. n-grams",
]
main_rows["model_order"] = main_rows["model_family"].map({name: i for i, name in enumerate(preferred_order)}).fillna(99)
main_rows = main_rows.sort_values(["model_order", "label_source", "political_f1"], ascending=[True, True, False])

main_table = main_rows[
    [
        "model_family",
        "training_labels",
        "train_rows",
        "threshold",
        "accuracy",
        "political_precision",
        "political_recall",
        "political_f1",
        "macro_f1",
        "weighted_f1",
    ]
].rename(
    columns={
        "model_family": "Model",
        "training_labels": "Labels",
        "train_rows": "N train",
        "threshold": "Thr.",
        "accuracy": "Acc.",
        "political_precision": "PC Prec.",
        "political_recall": "PC Rec.",
        "political_f1": "PC F1",
        "macro_f1": "Macro F1",
        "weighted_f1": "Wtd. F1",
    }
)

save_latex_table(
    main_table,
    "table_classifier_comparison_main.tex",
    "Validation performance of political-corruption classification models.",
    "tab:pc-classifier-comparison-main",
)
display(main_table)


In [ ]:
# Appendix table: all best-threshold political-corruption classification results.
appendix_model_table = inspection_best_results.copy()
appendix_model_table["Model"] = appendix_model_table.apply(model_label, axis=1)
appendix_model_table["Labels"] = appendix_model_table["label_source"].map(label_source_label)
appendix_model_table = appendix_model_table.sort_values(
    ["political_f1", "political_recall", "political_precision"],
    ascending=False,
)
appendix_model_table = appendix_model_table[
    [
        "comparison_run",
        "Model",
        "Labels",
        "train_rows",
        "threshold",
        "accuracy",
        "political_precision",
        "political_recall",
        "political_f1",
        "macro_f1",
        "weighted_f1",
        "predicted_positive_rate",
    ]
].rename(
    columns={
        "comparison_run": "Run",
        "train_rows": "N train",
        "threshold": "Thr.",
        "accuracy": "Acc.",
        "political_precision": "PC Prec.",
        "political_recall": "PC Rec.",
        "political_f1": "PC F1",
        "macro_f1": "Macro F1",
        "weighted_f1": "Wtd. F1",
        "predicted_positive_rate": "Pred. pos.",
    }
)

save_latex_table(
    appendix_model_table,
    "table_classifier_comparison_appendix.tex",
    "Full validation comparison of political-corruption classification models.",
    "tab:pc-classifier-comparison-appendix",
)
display(appendix_model_table)


In [ ]:
# Appendix table: country-level validation for the selected political-corruption classifier.
selected_predictions = inspection_prediction_results[
    inspection_prediction_results["embedding_model"].fillna("").eq(str(recommended["embedding_model"]))
    & inspection_prediction_results["label_source"].eq(recommended["label_source"])
    & inspection_prediction_results["comparison_run"].eq(recommended["comparison_run"])
].copy()

country_metric_rows = []
for country, group in selected_predictions.groupby("country"):
    y_true = group["y"].astype(int).to_numpy()
    y_pred = group["pred_best_threshold"].astype(int).to_numpy()
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], zero_division=0
    )
    _, _, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    country_metric_rows.append(
        {
            "Country": country.replace("United_Kingdom", "UK"),
            "N": len(group),
            "PC support": int((y_true == 1).sum()),
            "Acc.": accuracy_score(y_true, y_pred),
            "PC Prec.": precision[0],
            "PC Rec.": recall[0],
            "PC F1": f1[0],
            "Macro F1": macro_f1,
            "Wtd. F1": weighted_f1,
            "Pred. pos.": y_pred.mean(),
        }
    )

country_latex_table = pd.DataFrame(country_metric_rows).sort_values("PC F1", ascending=False)

save_latex_table(
    country_latex_table,
    "table_classifier_country_validation_appendix.tex",
    "Country-level validation performance of the selected political-corruption classifier.",
    "tab:pc-classifier-country-validation",
)
display(country_latex_table)


In [ ]:
# Appendix table: threshold sweep for the selected political-corruption classifier.
threshold_latex_table = candidate_thresholds[
    [
        "threshold",
        "accuracy",
        "political_precision",
        "political_recall",
        "political_f1",
        "macro_f1",
        "weighted_f1",
        "predicted_positive_rate",
    ]
].rename(
    columns={
        "threshold": "Thr.",
        "accuracy": "Acc.",
        "political_precision": "PC Prec.",
        "political_recall": "PC Rec.",
        "political_f1": "PC F1",
        "macro_f1": "Macro F1",
        "weighted_f1": "Wtd. F1",
        "predicted_positive_rate": "Pred. pos.",
    }
)

save_latex_table(
    threshold_latex_table,
    "table_classifier_threshold_sweep_appendix.tex",
    "Decision-threshold sweep for the selected political-corruption classifier.",
    "tab:pc-classifier-threshold-sweep",
)
display(threshold_latex_table)

print(f"\nLaTeX tables written to: {TABLE_DIR}")


## 8. Final Scoring Command

After deciding on the final model and threshold, run full-corpus scoring from the terminal. Pull the latest repo first if you use an E5 model, because `political_classifier/scripts/train_final_classifier.py` must apply the same E5 text prefix as the comparison script.

In [ ]:
print("Suggested final scoring command:\n")
print(
    "TMPDIR=/home/akroon/data/1t_storage/tmp \\n"
    "HF_HOME=/home/akroon/data/1t_storage/huggingface_cache \\n"
    "TRANSFORMERS_CACHE=/home/akroon/data/1t_storage/huggingface_cache \\n"
    "CUDA_VISIBLE_DEVICES=1 \\n"
    "nohup python3 -u political_classifier/scripts/train_final_classifier.py \\n"
    f"  --embedding-model {recommended['embedding_model']} \\n"
    f"  --threshold {recommended['threshold']} \\n"
    "  --extra-human-validation /home/akroon/data/1t_storage/RESPOND-victims-of-corruption/political_corruption_pipeline/active_learning/uk_human_validation_reviewed.csv \\n"
    "  --score-corpus \\n"
    "  > silver_classifier_final_scoring.log 2>&1 &"
)
